In [11]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import napari
import os
import tifffile
from napari.utils.colormaps import DirectLabelColormap
import matplotlib
from skimage.exposure import rescale_intensity

palette = ["red", "darkorange", "limegreen", "dodgerblue", "deeppink", "darkviolet"]
rgb_palette = [matplotlib.colors.to_rgb(c) for c in palette]
print(rgb_palette)

[(1.0, 0.0, 0.0), (1.0, 0.5490196078431373, 0.0), (0.19607843137254902, 0.803921568627451, 0.19607843137254902), (0.11764705882352941, 0.5647058823529412, 1.0), (1.0, 0.0784313725490196, 0.5764705882352941), (0.5803921568627451, 0.0, 0.8274509803921568)]


In [2]:
original_image = tifffile.imread(Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\ITERATIVE_LABEL_IMPROVEMENT\0_Bel_mCh_BF_VASCUMAP_FL33_ARi4_Original.tif"))

In [3]:
bf = original_image[:,0,:,:]
fluo = original_image[:,1,:,:]

In [4]:
labkit_fluo = tifffile.imread(Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\ITERATIVE_LABEL_IMPROVEMENT\1_Bel_mCh_BF_VASCUMAP_FL33_ARi4_Labkit_Combined_Fluo_BF.tif")).astype(bool)
labkit_bf_edges = tifffile.imread(Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\ITERATIVE_LABEL_IMPROVEMENT\1_Bel_mCh_BF_VASCUMAP_FL33_ARi4_Labkit_Edge_Only_BF.tif")).astype(bool)
labkit_bf_full = tifffile.imread(Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\ITERATIVE_LABEL_IMPROVEMENT\1_Bel_mCh_BF_VASCUMAP_FL33_ARi4_Labkit_From_BF.tif")).astype(bool)

In [5]:

manual_labels = tifffile.imread(Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\ITERATIVE_LABEL_IMPROVEMENT\2_Bel_mCh_BF_VASCUMAP_FL33_ARi4_initial_manual_labels.tif")).astype(bool)
v1_segmentation = tifffile.imread(Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\ITERATIVE_LABEL_IMPROVEMENT\3_Bel_mCh_BF_VASCUMAP_FL33_ARi4_v1_segmentation.tif")).astype(bool)
v2_segmentation = tifffile.imread(Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\ITERATIVE_LABEL_IMPROVEMENT\4_Bel_mCh_BF_VASCUMAP_FL33_ARi4_v2_segmentation.tif")).astype(bool)

In [ ]:
viewer = napari.Viewer()
scale = (9.997,1.13,1.13)
viewer.add_image(bf, name='Original Image (BF)', scale = scale)
viewer.add_image(fluo, name='Original Image (Fluo)', scale = scale)
labkit_fluo_layer = viewer.add_labels(labkit_fluo, scale = scale, name='Labkit Fluo')
labkit_fluo_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'gold'})




labkit_bf_edges_layer = viewer.add_labels(labkit_bf_edges, scale = scale, name='Labkit BF Edges')
labkit_bf_edges_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'darkorange'})
# viewer.add_labels(labkit_bf_edges, name='Labkit BF Edges')
labkit_bf_full_layer = viewer.add_labels(labkit_bf_full, scale = scale, name='Labkit BF Full')
labkit_bf_full_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'red'})
# viewer.add_labels(labkit_bf_full, name='Labkit BF Full')
manual_labels_layer = viewer.add_labels(manual_labels, scale = scale, name='Manual Labels')
manual_labels_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'limegreen'})
v1_layer = viewer.add_labels(v1_segmentation, scale = scale, name='V1 Segmentation')
v1_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'dodgerblue'})
v2_layer = viewer.add_labels(v2_segmentation, scale = scale, name='V2 Segmentation')
v2_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'deeppink'})
# viewer.add_labels(v1_segmentation, name='V1 Segmentation')
# viewer.add_labels(v2_segmentation, name='V2 Segmentation')

Rendering frames...


100%|██████████| 811/811 [01:52<00:00,  7.24it/s]


In [9]:
def tif_pixel_sizes(tif):
    """Return (z_um, y_um, x_um) from a tifffile TiffFile; any value is None if unavailable."""
    z_um = y_um = x_um = None
    tags = {t.name: t.value for t in tif.pages[0].tags.values()}
    xr = tags.get("XResolution")
    yr = tags.get("YResolution")
    if xr and xr[0]:
        x_um = xr[1] / xr[0]  # XResolution = pixels per unit -> um per pixel
    if yr and yr[0]:
        y_um = yr[1] / yr[0]
    ij = tif.imagej_metadata or {}
    if "spacing" in ij:
        z_um = float(ij["spacing"])
    return z_um, y_um, x_um

In [12]:
untreated_path_v2 = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260615_Exp7_Vascumap_BF_RFP_Thunder_DMSO_M1-7_DMSO_6channel.tif")
untreated_path_v1 = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V1_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260615_Exp7_Vascumap_BF_RFP_Thunder_DMSO_M1-7_DMSO_6channel.tif")
pax_path_v2 = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Paclitaxel_M2-3_Merged_6channel.tif")
pax_path_v1 = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V1_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Paclitaxel_M2-3_Merged_6channel.tif")
ibuprofen_path_v1 = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V1_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Ibuprofen_M1-8_Merged_6channel.tif")
ibuprofen_path_v2 = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\V2_UNTILTED_REPRESENTATIVE_IMAGE_OUTPUTS\20260611_Exp6_Vacumap_BF-RFP_Ibuprofen_M1-8_Merged_6channel.tif")
with tifffile.TiffFile(untreated_path_v2) as tif:
    z_um, y_um, x_um = tif_pixel_sizes(tif)
    print(f"Pixel sizes (z, y, x): {z_um}, {y_um}, {x_um}")



Pixel sizes (z, y, x): 9.99954, 1.3000000000605358, 1.3000000000605358


In [13]:
test = tifffile.imread(Path(untreated_path_v2))
v1 = tifffile.imread(Path(untreated_path_v1))
v1_seg = v1[:,-2,:,:].astype(np.uint8)
bf = test[:,0,:,:].astype(np.uint8)
fluo = test[:,1,:,:].astype(np.uint8)
v2_seg = test[:,-2,:,:].astype(np.uint8)
p_low, p_high = np.percentile(bf, (1, 99))
bf = rescale_intensity(bf, in_range=(p_low, p_high), out_range=(0, 255)).astype(np.uint8)
p_low, p_high = np.percentile(fluo, (1, 99))
fluo = rescale_intensity(fluo, in_range=(p_low, p_high), out_range=(0, 255)).astype(np.uint8)

MemoryError: Unable to allocate 22.6 GiB for an array with shape (6057836100,) and data type float32

In [ ]:
viewer = napari.Viewer()
scale = (z_um,x_um,y_um)
viewer.add_image(bf, name='Original Image (BF)', scale = scale)
viewer.add_image(fluo, name='Original Image (Fluo)', scale = scale)

v1_layer = viewer.add_labels(v1_seg, scale = scale, name='V1 Segmentation')
v1_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'dodgerblue'})
v2_layer = viewer.add_labels(v2_seg, scale = scale, name='V2 Segmentation')
v2_layer.colormap = DirectLabelColormap(color_dict={None: 'transparent', 1: 'deeppink'})